# SpatialMETA parameter benchmarking

## Imports

In [1]:
import seaborn as sns
import spatialmeta as smt
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import time
import psutil
import os

#from memory_profiler import memory_usage

## Loading data

In [2]:
joint_adata = smt.data.load_adata(
    sample_name="Y7_T_raw",
    modality="joint"
)

/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/site-packages/spatialmeta/data/./datasets/adata_joint_Y7_T_raw_raw.h5ad


In [3]:
type(joint_adata)

anndata._core.anndata.AnnData

## Preprocessing

In [4]:
process = psutil.Process(os.getpid())

mem_before = process.memory_info().rss / 1024**2
start = time.perf_counter()
####

joint_adata = smt.pp.removeHSP_MT_RPL_DNAJ(joint_adata)

joint_adata.layers["counts"] = joint_adata.X.copy()

smt.pp.normalize_total_joint_adata_sm_st(joint_adata,
                         target_sum_SM=1e4,
                         target_sum_ST=1e4)

joint_adata.layers["normalized"] = joint_adata.X.copy()

joint_adata.raw = joint_adata

smt.pp.spatial_variable_joint_adata_sm_st(joint_adata,
                                         n_top_genes = 2000,
                                         n_top_metabolites = 800,
                                         add_key = "highly_variable_moranI")

joint_adata = joint_adata[:,joint_adata.var.highly_variable_moranI]

joint_adata.write_h5ad("SpatialMETA/data/Y7_T_adata_joint_hvf2800.h5ad")

####
end = time.perf_counter()
mem_after = process.memory_info().rss / 1024**2

print(f"Runtime: {end-start:.2f} s")
print(f"Memory before: {mem_before:.2f} MB")
print(f"Memory after: {mem_after:.2f} MB")
print(f"Difference: {mem_after-mem_before:.2f} MB")

Runtime: 1.53 s
Memory before: 816.18 MB
Memory after: 1724.09 MB
Difference: 907.91 MB


## CVAE model

In [ ]:
process = psutil.Process(os.getpid())

mem_before = process.memory_info().rss / 1024**2
start = time.perf_counter()
####

joint_adata = sc.read_h5ad("SpatialMETA/data/Y7_T_adata_joint_hvf2800.h5ad")

joint_adata.X = joint_adata.layers["counts"]

smt.pp.normalize_total_joint_adata_sm_st(
    joint_adata,
    target_sum_SM=1e3,
    target_sum_ST=None
)

model = smt.model.ConditionalVAESTSM(
    joint_adata,
    device='cpu', # Small change to CPU instead of CUDA
    reconstruction_method_sm='g',
    reconstruction_method_st='zinb',
)

loss_dict = model.fit(
    max_epoch=64,
    lr=1e-3,
    mode='single'
)

####
end = time.perf_counter()
mem_after = process.memory_info().rss / 1024**2

print(f"Runtime: {end-start:.2f} s")
print(f"Memory before: {mem_before:.2f} MB")
print(f"Memory after: {mem_after:.2f} MB")
print(f"Difference: {mem_after-mem_before:.2f} MB")

####

Epoch 64: 100%|██████████| 64/64 [00:37<00:00,  1.71it/s, reconst_sm=-8.10e+01, reconst_st=3.25e+02, reconst_sm_corr=-8.60e+01, reconst_st_corr=3.24e+02, kldiv=4.68e+00, total_loss=3.08e+03, mmd_loss=0.00e+00]

Runtime: 37.74 s
Memory before: 1724.09 MB
Memory after: 1949.03 MB
Difference: 224.94 MB


## Visualisation

In [ ]:
Z = model.get_latent_embedding()
X = model.get_normalized_expression()
C = model.get_modality_contribution()

joint_adata.layers['reconstruction'] = X
joint_adata.obsm['X_emb']=Z
joint_adata.obs['contribution_st']=C
joint_adata.obs['contribution_sm']=1-C

sc.pp.neighbors(
    joint_adata,
    use_rep="X_emb",
    n_neighbors=15
)

sc.tl.umap(
    joint_adata,
    min_dist=1,
    spread=1
)

sc.tl.leiden(
    joint_adata,
    key_added="VAE_clusters_latent10"
)